# Nonstationary GP

入力位置によって latent function の smoothness が変わる場合に Gibbs kernel を使います。


## モデル構築


In [ ]:
import torch
from robotorchan.models import NonstationarySingleTaskGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
X = torch.linspace(0, 1, 16).unsqueeze(-1)
Y = torch.where(X < 0.5, torch.sin(2 * torch.pi * X), torch.sin(8 * torch.pi * X))
model = NonstationarySingleTaskGP(X, Y)
model.fit()


## 局所 lengthscale


In [ ]:
test_X = torch.linspace(0, 1, 5).unsqueeze(-1)
print(model.local_lengthscale(test_X))
print(model.posterior(test_X).mean.squeeze(-1))


## Nonstationary candidate proposal

非定常カーネルによる事後平均から候補を選び、元の目的関数で評価します。


In [ ]:
candidate_grid = torch.linspace(0, 1, 101).unsqueeze(-1)
model.eval()
with torch.no_grad():
    scores = model.posterior(candidate_grid).mean.squeeze(-1)
candidate = candidate_grid[scores.argmax().item():scores.argmax().item() + 1]
candidate_Y = torch.where(
    candidate < 0.5,
    torch.sin(2 * torch.pi * candidate),
    torch.sin(8 * torch.pi * candidate),
)
assert candidate.shape == (1, 1)
print("candidate:", candidate, "evaluated objective:", candidate_Y)
